In [ ]:
%config Completer.use_jedi = False

: 

In [1]:
from embed.embedder import Embedder

embed = Embedder()

q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

2026-06-22 05:35:42.438815698 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [2]:
v1.dot(dv)

np.float64(0.3233238425677314)

In [ ]:
v2.dot(dv)

np.float64(0.01973045838992233)

In [19]:
from ingest import load_faq_data

documents = load_faq_data()

In [20]:
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

In [21]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

/workspaces/LLM-Zoomcamp-2026/Module2: Embedding/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 27/27 [01:00<00:00,  2.25s/it]


In [22]:
query = "Can I still join the course after the start date?"
v_query = embed.encode(query)

scores = X.dot(v_query)
idx = np.argmax(scores)

documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [16]:
#HW2
q1 = 'How does approximate nearest neighbor search work?'
e1 = embed.encode(q1)

In [34]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [7]:
len(documents)

72

In [14]:
doc22 = documents[22]
doc22

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [13]:
texts = doc22["content"]
texts

'# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done so far is ex

In [17]:
e_doc22 = embed.encode(texts)

In [19]:
print(e_doc22.dot(e1))

0.36107027225589694


In [20]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [22]:
texts = [chunk["content"] for chunk in chunks]

In [23]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

/workspaces/LLM-Zoomcamp-2026/Module2: Embedding/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 6/6 [00:12<00:00,  2.08s/it]


In [27]:
scores = X.dot(e1)
idx = np.argmax(scores)

chunks[idx]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

In [29]:
idx, scores[idx]

(np.int64(94), np.float64(0.6489017718578813))

In [43]:
q2='What metric do we use to evaluate a search engine?'
e2 = embed.encode(q2)
e2

array([-3.45251120e-02, -4.76347907e-02, -9.52288143e-02,  2.40308090e-02,
       -1.80039094e-02,  3.22336786e-02,  4.11009181e-04,  2.86892654e-02,
        2.71596053e-02, -6.36475643e-02, -3.86707619e-02, -5.52097101e-02,
        3.84640827e-02,  3.63455416e-02, -9.32710316e-02, -5.29945155e-02,
        8.20461574e-02, -5.03414745e-03,  2.50910438e-02, -8.70656696e-02,
        7.20100754e-02,  1.29639035e-02,  1.16451658e-01, -8.20982256e-02,
       -6.52023347e-03, -4.00227584e-02, -8.36716391e-02, -1.24123525e-02,
       -1.90502106e-02, -6.24077402e-03, -3.73876830e-02,  6.12152591e-03,
        5.60926979e-02,  6.56479763e-02, -6.34850137e-02,  3.43527274e-02,
       -3.35941763e-02, -3.51476658e-02,  2.54281298e-02, -9.75776093e-03,
       -4.84886223e-02, -2.35025387e-02, -2.90664668e-02,  3.65187041e-02,
        2.23322831e-02, -3.29535923e-02, -4.82611746e-02,  8.59540277e-03,
       -3.70969933e-03,  5.49367302e-02, -9.72623872e-02, -1.93188061e-04,
       -1.25472844e-02, -

In [55]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [56]:
texts = [doc["content"] for doc in documents]

In [57]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:03<00:00,  1.58s/it]


In [66]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, documents)

In [49]:
vindex.search(e2, num_results=1)

[{'content': '# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup, each query

In [50]:
scores = X.dot(e2)
idx = np.argmax(scores)

documents[idx]

{'content': '# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup, each query 

In [60]:
q3='How do I store vectors in PostgreSQL?'
e3=embed.encode(q3)

In [61]:
vindex.search(e3, num_results=5)

[{'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWORD=pswd \\\n  

In [64]:
from minsearch import Index

index = Index(
    text_fields=["content"]
    )

index.fit(documents)

In [54]:

search_results = index.search(
    q3,
    num_results=5
)

search_results

[{'content': '# Embeddings\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=kJOlW1HeMp4&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nBefore we can do vector search, we need to turn our text into vectors.\nWe call this process embedding: we embed text into a vector space. The\nvectors we get back are also called "embeddings."\n\n## Word embeddings and sentence embeddings\n\nThis idea comes from\n[word2vec](https://en.wikipedia.org/wiki/Word2vec). The model learns to\nplace words as points in a multi-dimensional space. Words with similar\nmeanings land close to each other.\n\nImagine a 2D space where "enroll" and "join" are near each other and\n"Docker" is far away:\n\n```text\n        · enroll\n       · join\n                   · Docker\n```\n\nThe same idea works for entire sentences:\n\n```text\nQ1: "I just discovered the course. Can I still join it?"\nQ2: "I just found out about the program. Can I still enroll?"\n\nThese two are close - they mean the same thing.\n\nQ3: "H

In [62]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [63]:
q4='How do I give the model access to tools?'
e4 = embed.encode(q4)

In [65]:

text_results = index.search(
    q4,
    num_results=5
)

text_results

[{'content': '# Function Calling\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=CeEki_0mdGo&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson we built a RAG pipeline with `RAGBase.rag()`\nand saw it fail on the "Olama" typo. The search returned nothing\nuseful, and the LLM had no way to recover.\n\nThe flow that broke:\n\n```mermaid\nflowchart TD\n    U([User: How do I run Olama?])\n    S[search - Olama - no useful results]\n    A([LLM: I don\'t have information about Olama.])\n\n    U --> S --> A\n```\n\nThe pipeline is fixed: search, build prompt, LLM.\n\n```python\ndef rag(self, query):\n    search_results = self.search(query)\n    prompt = self.build_prompt(query, search_results)\n    answer = self.llm(prompt)\n    return answer\n```\n\nThe LLM is a passenger here, not a driver. It never sees the bad\nsearch results, so it can\'t try again with a corrected query.\n\n## The agent alternative\n\nAn agent puts the LLM in charge.\n\nInstead of running se

In [69]:
vector_results = vindex.search(e4, num_results=5)
vector_results

[{'content': "# Other Frameworks\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=4yiCbKX9RhI&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nThe concepts you learned in Part 2 are the same across frameworks.\nFunction calling, the agent loop, and tool definitions all wrap the\nsame pattern. Send messages, run any function calls, and repeat until\nthe model is done.\n\nYou now understand how the loop works. So you can pick up any\nproduction framework and know what it's doing under the hood. I kept\nthis module framework agnostic on purpose, so you can explore and pick\nthe one you like.\n\nHere are some frameworks worth a look:\n\n## OpenAI Agents SDK\n\nThe official SDK from OpenAI for building agents. It uses the same\nResponses API we used throughout this module. It supports tool\ndefinition, multi-turn conversations, and handoffs between agents.\n\n```bash\nuv add openai-agents\n```\n\nGood choice if you're already using OpenAI and want something\nofficial and well-mainta

In [68]:
results = rrf([vector_results, text_results])

KeyError: 'start'